# 04. 모델 C — 판매 트렌드 분석 (Pandas)
딥러닝 모델 없이 판매 데이터만으로 트렌드 점수를 계산합니다.
- **트렌드 점수** = 7일판매(60%) + 30일판매(30%) + 조회수(10%)
- 상품 후보를 받아 트렌드 점수 순으로 정렬해 반환합니다.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
W_7D   = 0.6
W_30D  = 0.3
W_VIEW = 0.1

def load_data():
    products = pd.read_csv('data/상품목록.csv', encoding='utf-8-sig')
    sales    = pd.read_csv('data/판매데이터.csv', encoding='utf-8-sig')
    return products.merge(sales, on='product_id')

def compute_trend_score(df):
    def minmax(col):
        mn, mx = col.min(), col.max()
        return (col - mn) / (mx - mn + 1e-8)
    df = df.copy()
    df['trend_score'] = (
        W_7D   * minmax(df['sales_7d']) +
        W_30D  * minmax(df['sales_30d']) +
        W_VIEW * minmax(df['view_count'])
    )
    return df

def rank_candidates(candidate_ids):
    """모델 A·B가 필터링한 상품 ID → 트렌드 점수 순 DataFrame 반환"""
    df = compute_trend_score(load_data())
    return (
        df[df['product_id'].isin(candidate_ids)]
        .sort_values('trend_score', ascending=False)
        [['product_id', 'product_name', 'style', 'size',
          'price', 'sales_7d', 'sales_30d', 'trend_score']]
    )

In [ ]:
# 전체 트렌드 TOP 10
df = compute_trend_score(load_data())
print('▶ 전체 트렌드 TOP 10')
df.nlargest(10, 'trend_score')[['product_name', 'style', 'size', 'sales_7d', 'trend_score']]

In [ ]:
# 스타일별 트렌드 1위
print('▶ 스타일별 트렌드 1위')
(
    df.sort_values('trend_score', ascending=False)
    .groupby('style').first().reset_index()
    [['style', 'product_name', 'trend_score']]
    .sort_values('trend_score', ascending=False)
)

In [ ]:
# 사이즈별 30일 판매량 합계
print('▶ 사이즈별 판매량 합계 (30일)')
df.groupby('size')['sales_30d'].sum().sort_values(ascending=False).to_frame()